# Heart Failure Prediction - Complete ML Pipeline with Advanced Visualizations

## Complete ML Workflow with Comprehensive Model Evaluation Visualizations

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report, roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## Step 2: Load and Explore Data

In [ ]:
df = pd.read_csv('heart.csv')
print('Dataset Shape:', df.shape)
print('Missing Values:', df.isnull().sum().sum())
print('Duplicate Rows:', df.duplicated().sum())
print('\nTarget Distribution:')
print(df['HeartDisease'].value_counts())

## Step 3: Data Preprocessing

In [ ]:
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

for col in ['Cholesterol', 'RestingBP']:
    median_val = X[X[col] != 0][col].median()
    X[col] = X[col].replace(0, median_val)

scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
print('Data preprocessing completed')

## Step 4: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## Step 5: Train All Models

In [ ]:
models = {
    'SVM (Linear)': SVC(kernel='linear', probability=True, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'LDA': LinearDiscriminantAnalysis()
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
test_results = {}
confusion_matrices = {}
scoring = ['accuracy', 'roc_auc', 'recall', 'precision', 'f1']
predictions = {}

for model_name, model in models.items():
    print(f'Training {model_name}...')
    cv_scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    
    cv_results[model_name] = {
        'Accuracy': cv_scores['test_accuracy'].mean(),
        'ROC-AUC': cv_scores['test_roc_auc'].mean(),
        'Recall': cv_scores['test_recall'].mean(),
        'Precision': cv_scores['test_precision'].mean(),
        'F1': cv_scores['test_f1'].mean(),
    }
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    test_results[model_name] = {
        'Test_Accuracy': accuracy_score(y_test, y_pred),
        'Test_ROC_AUC': roc_auc_score(y_test, y_pred_proba),
        'Test_Recall': recall_score(y_test, y_pred),
        'Test_Precision': precision_score(y_test, y_pred),
    }
    
    confusion_matrices[model_name] = confusion_matrix(y_test, y_pred)
    predictions[model_name] = {'y_pred': y_pred, 'y_pred_proba': y_pred_proba}
    print(f'  Test ROC-AUC: {test_results[model_name]["Test_ROC_AUC"]:.4f}')

## Step 6: CONFUSION MATRICES FOR ALL MODELS - HEATMAP VISUALIZATION

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

model_names = list(confusion_matrices.keys())

for idx, model_name in enumerate(model_names):
    cm = confusion_matrices[model_name]
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, ax=axes[idx],
                xticklabels=['No Disease', 'Heart Failure'],
                yticklabels=['No Disease', 'Heart Failure'],
                cbar_kws={'label': 'Count'})
    
    accuracy = test_results[model_name]['Test_Accuracy']
    roc_auc = test_results[model_name]['Test_ROC_AUC']
    
    axes[idx].set_title(f'{model_name}\nAccuracy: {accuracy:.3f} | ROC-AUC: {roc_auc:.3f}',
                        fontweight='bold', fontsize=11)
    axes[idx].set_ylabel('True Label', fontsize=10)
    axes[idx].set_xlabel('Predicted Label', fontsize=10)

axes[5].axis('off')
axes[5].text(0.1, 0.5, 'CONFUSION MATRIX METRICS\n\nTN = True Negative\nFP = False Positive\nFN = False Negative\nTP = True Positive',
            fontsize=11, family='monospace', verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Confusion Matrices - All Models', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## Step 7: Model Performance Metrics Heatmap

In [ ]:
test_df = pd.DataFrame(test_results).T
cv_df = pd.DataFrame(cv_results).T

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(cv_df, annot=True, fmt='.4f', cmap='RdYlGn', cbar=True, ax=axes[0],
            cbar_kws={'label': 'Score'})
axes[0].set_title('Cross-Validation Metrics Heatmap', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Metrics', fontsize=11)
axes[0].set_ylabel('Models', fontsize=11)

sns.heatmap(test_df, annot=True, fmt='.4f', cmap='RdYlGn', cbar=True, ax=axes[1],
            cbar_kws={'label': 'Score'})
axes[1].set_title('Test Set Metrics Heatmap', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Metrics', fontsize=11)
axes[1].set_ylabel('Models', fontsize=11)

plt.tight_layout()
plt.show()

## Step 8: Detailed Confusion Matrix Analysis

In [ ]:
print('='*80)
print('CONFUSION MATRIX ANALYSIS FOR ALL MODELS')
print('='*80)

for model_name in model_names:
    cm = confusion_matrices[model_name]
    tn, fp, fn, tp = cm.ravel()
    
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    print(f'\n{model_name}:')
    print(f'  True Negatives:  {tn:3d}  |  False Positives: {fp:3d}')
    print(f'  False Negatives: {fn:3d}  |  True Positives:  {tp:3d}')
    print(f'  Sensitivity (Recall):  {sensitivity:.4f}  |  Specificity: {specificity:.4f}')
    print(f'  Accuracy: {test_results[model_name]["Test_Accuracy"]:.4f}  |  ROC-AUC: {test_results[model_name]["Test_ROC_AUC"]:.4f}')

## Step 9: ROC Curves for All Models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['blue', 'orange', 'green', 'red', 'purple']

for idx, (model_name, color) in enumerate(zip(model_names, colors)):
    y_pred_proba = predictions[model_name]['y_pred_proba']
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=color, lw=2.5, label=f'{model_name} (AUC={roc_auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate', fontsize=11)
axes[0].set_ylabel('True Positive Rate', fontsize=11)
axes[0].set_title('ROC Curves - All Models', fontweight='bold', fontsize=12)
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(alpha=0.3)

# Model comparison bar chart
model_auc_scores = [test_results[model]['Test_ROC_AUC'] for model in model_names]
model_acc_scores = [test_results[model]['Test_Accuracy'] for model in model_names]

x_pos = np.arange(len(model_names))
width = 0.35

axes[1].bar(x_pos - width/2, model_auc_scores, width, label='ROC-AUC', color='steelblue')
axes[1].bar(x_pos + width/2, model_acc_scores, width, label='Accuracy', color='orange')
axes[1].set_xlabel('Models', fontsize=11)
axes[1].set_ylabel('Score', fontsize=11)
axes[1].set_title('Model Performance Comparison', fontweight='bold', fontsize=12)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(model_names, rotation=45, ha='right')
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim([0.75, 0.95])

plt.tight_layout()
plt.show()

## Step 10: Best Model Selection & Summary

In [ ]:
best_model_name = cv_df['ROC-AUC'].idxmax()
best_cv_auc = cv_df.loc[best_model_name, 'ROC-AUC']
best_test_auc = test_df.loc[best_model_name, 'Test_ROC_AUC']
best_cm = confusion_matrices[best_model_name]

print('\n' + '='*80)
print('BEST MODEL SELECTED')
print('='*80)
print(f'\nModel: {best_model_name}')
print(f'\nCross-Validation Performance:')
print(f'  ROC-AUC: {best_cv_auc:.4f}')
print(f'  Accuracy: {cv_df.loc[best_model_name, "Accuracy"]:.4f}')
print(f'  Recall: {cv_df.loc[best_model_name, "Recall"]:.4f}')
print(f'\nTest Set Performance:')
print(f'  ROC-AUC: {best_test_auc:.4f}')
print(f'  Accuracy: {test_df.loc[best_model_name, "Test_Accuracy"]:.4f}')
print(f'  Recall: {test_df.loc[best_model_name, "Test_Recall"]:.4f}')
print(f'  Precision: {test_df.loc[best_model_name, "Test_Precision"]:.4f}')

print(f'\nConfusion Matrix:')
print(best_cm)
tn, fp, fn, tp = best_cm.ravel()
print(f'  TN: {tn} | FP: {fp} | FN: {fn} | TP: {tp}')

## Step 11: Final Summary

In [ ]:
print('\n' + '='*80)
print('COMPLETE ML PIPELINE SUMMARY')
print('='*80)

summary_text = f'''\nDataset Information:
  • Total Samples: 918
  • Features: 11 clinical measurements
  • Classes: 2 (No Disease vs Heart Failure)
  • Train/Test Split: 80/20 (Stratified)

Models Evaluated:
  1. SVM (Linear) - ROC-AUC: {test_results['SVM (Linear)']['Test_ROC_AUC']:.4f}
  2. SVM (RBF) - ROC-AUC: {test_results['SVM (RBF)']['Test_ROC_AUC']:.4f}
  3. Random Forest - ROC-AUC: {test_results['Random Forest']['Test_ROC_AUC']:.4f}
  4. Logistic Regression - ROC-AUC: {test_results['Logistic Regression']['Test_ROC_AUC']:.4f}
  5. LDA - ROC-AUC: {test_results['LDA']['Test_ROC_AUC']:.4f}

BEST MODEL: {best_model_name}
  • CV ROC-AUC: {best_cv_auc:.4f}
  • Test ROC-AUC: {best_test_auc:.4f}
  • Test Accuracy: {test_df.loc[best_model_name, 'Test_Accuracy']:.4f}
  • Test Recall: {test_df.loc[best_model_name, 'Test_Recall']:.4f}

Visualizations Generated:
  ✓ Confusion Matrices (5 models as heatmaps)
  ✓ Performance Metrics Heatmap (CV vs Test)
  ✓ ROC Curves (All models)
  ✓ Model Comparison Charts
  ✓ Detailed Confusion Matrix Analysis

Status: READY FOR PRODUCTION DEPLOYMENT!
'''

print(summary_text)